# Python Decorators and Function Wrappers

Decorators are a powerful and advanced feature in Python that allows you to modify or extend the behavior of functions and methods without changing their implementation. This notebook provides a comprehensive exploration of decorators, from basic concepts to advanced applications.

## Understanding Functions as First-Class Objects

In Python, functions are "first-class objects," which means they can be:
- Assigned to variables
- Passed as arguments to other functions
- Returned from functions
- Stored in data structures

This property is what makes decorators possible.

In [ ]:
# Functions as first-class objects

# 1. Assign a function to a variable
def greet(name):
    return f"Hello, {name}!"

say_hello = greet  # No parentheses, we're assigning the function itself, not its result
print(say_hello("Alice"))  # Uses the function via the new name

# 2. Pass a function as an argument
def execute_function(func, arg):
    return func(arg)

result = execute_function(greet, "Bob")
print(result)

# 3. Return a function from another function
def get_greeting_function(prefix):
    def custom_greeting(name):
        return f"{prefix}, {name}!"
    return custom_greeting

casual_greeting = get_greeting_function("Hey")
formal_greeting = get_greeting_function("Good day")

print(casual_greeting("Charlie"))
print(formal_greeting("Diana"))

# 4. Store functions in data structures
greeting_funcs = [casual_greeting, formal_greeting]
for func in greeting_funcs:
    print(func("Eve"))

## Basic Decorator Syntax

A decorator is a function that takes another function as input and extends or modifies its behavior without explicitly changing its code.

The basic syntax uses the `@` symbol placed above the function definition.

In [ ]:
# Basic decorator example

def simple_decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()  # Call the original function
        print("Something is happening after the function is called.")
    return wrapper

# Method 1: Using the @ syntax (syntactic sugar)
@simple_decorator
def say_hello():
    print("Hello!")

say_hello()

print("\n" + "-" * 40 + "\n")

# Method 2: Equivalent to the @ syntax, showing what happens behind the scenes
def say_goodbye():
    print("Goodbye!")

decorated_say_goodbye = simple_decorator(say_goodbye)
decorated_say_goodbye()

## Creating Simple Decorators

Let's create some practical decorators for common use cases:
1. A timing decorator to measure function execution time
2. A logging decorator to log function calls
3. A validation decorator to check function inputs

In [ ]:
import time
import logging
import functools

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 1. Timing decorator
def timer_decorator(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} executed in {end_time - start_time:.4f} seconds")
        return result
    return wrapper

# 2. Logging decorator
def log_decorator(func):
    def wrapper(*args, **kwargs):
        logging.info(f"Calling {func.__name__} with args: {args}, kwargs: {kwargs}")
        result = func(*args, **kwargs)
        logging.info(f"{func.__name__} returned: {result}")
        return result
    return wrapper

# 3. Validation decorator (for numeric arguments)
def validate_numeric(func):
    def wrapper(*args, **kwargs):
        for arg in args:
            if not isinstance(arg, (int, float)):
                raise TypeError(f"Argument {arg} is not a number")
        for key, value in kwargs.items():
            if not isinstance(value, (int, float)):
                raise TypeError(f"Argument {key}={value} is not a number")
        return func(*args, **kwargs)
    return wrapper

# Example usage

@timer_decorator
def factorial(n):
    """Calculate the factorial of n"""
    if n <= 1:
        return 1
    else:
        return n * factorial(n-1)

@log_decorator
def add(a, b):
    """Add two numbers"""
    return a + b

@validate_numeric
def divide(a, b):
    """Divide a by b"""
    return a / b

# Test the decorators
print("Testing timer_decorator:")
result = factorial(10)
print(f"Result: {result}\n")

print("Testing log_decorator:")
result = add(3, 4)
print(f"Result: {result}\n")

print("Testing validate_numeric:")
try:
    result = divide(10, 2)
    print(f"Result: {result}")
    
    result = divide(10, "2")  # This should raise an error
    print(f"Result: {result}")
except TypeError as e:
    print(f"Error caught: {e}")

## Preserving Function Metadata with functools.wraps

When you create a decorator, the wrapped function loses its original metadata like name, docstring, and argument list. The `functools.wraps` decorator helps preserve this information.

In [ ]:
import functools

# Without using functools.wraps
def decorator_without_wraps(func):
    def wrapper(*args, **kwargs):
        """This is the wrapper function"""
        return func(*args, **kwargs)
    return wrapper

# With using functools.wraps
def decorator_with_wraps(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        """This is the wrapper function"""
        return func(*args, **kwargs)
    return wrapper

# Example functions
@decorator_without_wraps
def example_function1():
    """This is the docstring for example_function1"""
    pass

@decorator_with_wraps
def example_function2():
    """This is the docstring for example_function2"""
    pass

# Compare the metadata
print("Without functools.wraps:")
print(f"Function name: {example_function1.__name__}")
print(f"Docstring: {example_function1.__doc__}")
print(f"Module: {example_function1.__module__}")

print("\nWith functools.wraps:")
print(f"Function name: {example_function2.__name__}")
print(f"Docstring: {example_function2.__doc__}")
print(f"Module: {example_function2.__module__}")

# Improved timer decorator with wraps
def better_timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} executed in {end_time - start_time:.4f} seconds")
        return result
    return wrapper

@better_timer
def calculate_sum(n):
    """Calculate the sum of numbers from 1 to n"""
    return sum(range(1, n + 1))

print("\nImproved timer decorator example:")
result = calculate_sum(1000000)
print(f"Function name: {calculate_sum.__name__}")
print(f"Docstring: {calculate_sum.__doc__}")
print(f"Result: {result}")

## Decorators with Arguments

We can create more flexible decorators by adding parameters to them. This requires an additional level of function nesting.

In [ ]:
# Decorator with arguments
def repeat(num_times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(num_times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))

# Rate limiter decorator with arguments
def rate_limit(max_calls, period):
    """Limit the rate of function calls.
    
    Args:
        max_calls: Maximum number of calls allowed in the period
        period: Time period in seconds
    """
    call_timestamps = []
    
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            current_time = time.time()
            
            # Remove timestamps older than the period
            while call_timestamps and call_timestamps[0] < current_time - period:
                call_timestamps.pop(0)
            
            # Check if we've reached the limit
            if len(call_timestamps) >= max_calls:
                raise Exception(f"Rate limit exceeded. Maximum {max_calls} calls allowed per {period} seconds.")
            
            # Add current timestamp and call the function
            call_timestamps.append(current_time)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=3, period=1)
def limited_function():
    return "Function was called"

# Test the rate limiter
try:
    print("\nTesting rate limiter:")
    for i in range(5):
        print(f"Call {i+1}: {limited_function()}")
except Exception as e:
    print(f"Error: {e}")

## Multiple Decorators

You can apply multiple decorators to a single function. The decorators are applied from the innermost (closest to the function) to the outermost.

In [ ]:
# Example with multiple decorators

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

def underline(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<u>{func(*args, **kwargs)}</u>"
    return wrapper

# Apply multiple decorators
@bold
@italic
@underline
def format_text(text):
    return text

# Test with different order
@underline
@italic
@bold
def format_text2(text):
    return text

print("Multiple decorators example:")
print(format_text("Hello, World!"))  # Should apply bold -> italic -> underline
print(format_text2("Hello, World!"))  # Should apply underline -> italic -> bold

# Visual explanation of execution order
print("\nExecution order:")
print("1. @bold @italic @underline def func(): ...")
print("   Executes as: bold(italic(underline(func)))()")

# Decorators with logging to show execution order
def log_decorator1(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Entering decorator 1 for {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Exiting decorator 1 for {func.__name__}")
        return result
    return wrapper

def log_decorator2(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Entering decorator 2 for {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Exiting decorator 2 for {func.__name__}")
        return result
    return wrapper

@log_decorator1
@log_decorator2
def example_function():
    print("This is the example function executing")
    return "Function result"

print("\nLogging decorators to show execution order:")
result = example_function()
print(f"Final result: {result}")

## Class Decorators

Decorators can also be applied to classes to modify their behavior. Additionally, we can create decorators using classes (instead of functions) that have a `__call__` method.

In [ ]:
# 1. Decorating a class
def add_greeting(cls):
    cls.greet = lambda self: f"Hello, I'm a {self.__class__.__name__}"
    return cls

@add_greeting
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def __str__(self):
        return f"{self.name}, {self.age} years old"

# 2. Creating a decorator as a class
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.num_calls = 0
    
    def __call__(self, *args, **kwargs):
        self.num_calls += 1
        print(f"{self.func.__name__} has been called {self.num_calls} times")
        return self.func(*args, **kwargs)

@CountCalls
def say_whatsup():
    return "What's up?"

# Test the class decoration
print("Class decorator example:")
person = Person("Alice", 30)
print(person)
print(person.greet())

# Test the decorator class
print("\nDecorator class example:")
print(say_whatsup())
print(say_whatsup())
print(say_whatsup())

# Singleton pattern using a class decorator
def singleton(cls):
    instances = {}
    
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    
    return get_instance

@singleton
class Database:
    def __init__(self, name):
        print(f"Creating a new database connection to {name}")
        self.name = name
        
    def query(self, sql):
        return f"Executing '{sql}' on database '{self.name}'"

# Test singleton
print("\nSingleton decorator example:")
db1 = Database("MySQL")  # Should print the creation message
db2 = Database("PostgreSQL")  # Should NOT print the creation message
print(f"db1 name: {db1.name}")
print(f"db2 name: {db2.name}")  # Should be the same as db1.name
print(f"Are db1 and db2 the same object? {db1 is db2}")

## Practical Use Cases for Decorators

Decorators have many practical applications in real-world Python programming. Here are some common use cases:

In [ ]:
# 1. Memoization (caching results)
def memoize(func):
    cache = {}
    
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Convert args and kwargs to a hashable key
        key = str(args) + str(sorted(kwargs.items()))
        
        if key not in cache:
            print(f"Computing {func.__name__}{args}, {kwargs}")
            cache[key] = func(*args, **kwargs)
        else:
            print(f"Fetching from cache for {func.__name__}{args}, {kwargs}")
            
        return cache[key]
    
    return wrapper

@memoize
def fibonacci(n):
    """Compute the Fibonacci number recursively."""
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# 2. Retry pattern
def retry(max_attempts, delay=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts == max_attempts:
                        raise
                    print(f"Attempt {attempts} failed: {e}. Retrying in {delay} seconds...")
                    time.sleep(delay)
            return None
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.1)
def unstable_function(success_rate=0.3):
    """A function that fails randomly for demo purposes."""
    import random
    if random.random() > success_rate:
        raise RuntimeError("Function failed randomly")
    return "Function succeeded"

# 3. Authentication decorator
def require_auth(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # In a real app, this would check session, tokens, etc.
        auth_token = kwargs.get('auth_token')
        if auth_token != 'valid_token':
            raise PermissionError("Authentication required")
        return func(*args, **kwargs)
    return wrapper

@require_auth
def get_sensitive_data(user_id, auth_token=None):
    return f"Sensitive data for user {user_id}"

# Test the practical use cases
print("Memoization example:")
print(f"fibonacci(6) = {fibonacci(6)}")
print(f"fibonacci(6) again = {fibonacci(6)}")  # Should use cache
print()

print("Retry example:")
try:
    result = unstable_function()
    print(f"Result: {result}")
except Exception as e:
    print(f"Final error: {e}")
print()

print("Authentication example:")
try:
    print(get_sensitive_data(123, auth_token="invalid"))
except Exception as e:
    print(f"Error with invalid token: {e}")
    
try:
    print(get_sensitive_data(123, auth_token="valid_token"))
except Exception as e:
    print(f"Error: {e}")

## Common Decorator Patterns

Here are some common patterns and best practices for writing decorators.

In [ ]:
# 1. Decorator template pattern
def decorator_template(func=None, *, keyword_arg1=None):
    def actual_decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Do something before
            result = func(*args, **kwargs)
            # Do something after
            return result
        return wrapper
    
    # This allows the decorator to be used with or without arguments
    if func is None:
        return actual_decorator
    else:
        return actual_decorator(func)

# 2. Stateful decorators
def counter(start=0):
    def actual_decorator(func):
        # Using nonlocal is one way to maintain state
        nonlocal start
        count = start
        
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal count
            count += 1
            print(f"Call count: {count}")
            return func(*args, **kwargs)
        
        # Another way is to attach attributes to the wrapper function
        wrapper.count = lambda: count
        wrapper.reset = lambda: setattr(wrapper, 'count', lambda: start)
        
        return wrapper
    return actual_decorator

@counter(start=10)
def example_function():
    return "Function called"

# 3. Parameterized class decorator
class Debuggable:
    def __init__(self, debug=True):
        self.debug = debug
    
    def __call__(self, cls):
        original_init = cls.__init__
        
        @functools.wraps(original_init)
        def new_init(self_obj, *args, **kwargs):
            if self.debug:
                print(f"Initializing {cls.__name__} with {args} and {kwargs}")
            original_init(self_obj, *args, **kwargs)
            
        cls.__init__ = new_init
        return cls

@Debuggable(debug=True)
class Customer:
    def __init__(self, name, account_type):
        self.name = name
        self.account_type = account_type

# Test decorator patterns
print("Stateful decorator example:")
print(example_function())
print(example_function())
print(f"Current count: {example_function.count()}")
example_function.reset()
print(f"After reset count: {example_function.count()}")
print(example_function())
print()

print("Class decorator with parameters:")
customer = Customer("John Doe", "Premium")

## Summary

In this notebook, we've explored Python decorators and function wrappers in depth:

1. **Functions as First-Class Objects**: Understanding Python's treatment of functions as objects
2. **Basic Decorator Syntax**: How to create and apply simple decorators
3. **Creating Simple Decorators**: Practical examples like timing, logging, and validation
4. **Preserving Metadata**: Using `functools.wraps` to maintain function metadata
5. **Decorators with Arguments**: Creating flexible decorators that accept parameters
6. **Multiple Decorators**: Applying and understanding execution order of multiple decorators
7. **Class Decorators**: Using decorators with classes and creating decorators as classes
8. **Practical Use Cases**: Real-world applications like memoization, retry logic, and authentication
9. **Common Patterns**: Best practices and patterns for writing effective decorators

Decorators are a powerful feature in Python that enable clean, reusable, and maintainable code by applying cross-cutting concerns without modifying the original function's code.